# SC-SSTW real-Wan carrier propagation v2
Runs only the repository v2 CLI. Paired RGB/MP4 evidence is diagnostic; the blind relation uses one saved watermarked MP4.


In [ ]:
# 1. Mount Drive and initialize an immutable v2 run
from google.colab import drive, userdata
from pathlib import Path
import datetime, os
drive.mount('/content/drive')
BOOTSTRAP_ERROR = None
def _secret_or_none(name):
    try:
        return userdata.get(name)
    except Exception:
        return None
EXPECTED_COMMIT = (os.environ.get('SC_SSTW_COMMIT') or _secret_or_none('SC_SSTW_COMMIT') or '').strip()
HF_TOKEN = os.environ.get('HF_TOKEN') or _secret_or_none('HF_TOKEN')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
if len(EXPECTED_COMMIT) != 40:
    BOOTSTRAP_ERROR = 'SC_SSTW_COMMIT must be an exact 40-character commit'
REPO_URL = os.environ.get('SC_SSTW_REPO_URL', 'https://github.com/RICHAAARC/SC-SSTW-Feasibility.git')
RUN_STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_NAME = f"{RUN_STAMP}_{EXPECTED_COMMIT[:8] if len(EXPECTED_COMMIT) == 40 else 'invalidcommit'}"
LOCAL_RUN = Path('/content') / RUN_NAME
DRIVE_PARENT = Path('/content/drive/MyDrive/SC-SSTW-Feasibility/runs/gpu_carrier_propagation_v2')
print({'commit': EXPECTED_COMMIT, 'run_name': RUN_NAME, 'bootstrap_error': BOOTSTRAP_ERROR})


In [ ]:
# 2. Fetch and verify the exact clean checkout
import subprocess
REPO = Path('/content/SC-SSTW-Feasibility')
if BOOTSTRAP_ERROR is None:
    try:
        if not REPO.exists():
            subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO)], check=True)
        subprocess.run(['git', 'fetch', 'origin', EXPECTED_COMMIT], cwd=REPO, check=True)
        subprocess.run(['git', 'checkout', '--detach', EXPECTED_COMMIT], cwd=REPO, check=True)
        observed = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
        dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip()
        if observed != EXPECTED_COMMIT or dirty:
            raise RuntimeError(f'checkout mismatch or dirty state: observed={observed}, dirty={dirty!r}')
    except Exception as exc:
        BOOTSTRAP_ERROR = f'checkout failure: {type(exc).__name__}: {exc}'
print({'repository': REPO_URL, 'commit': globals().get('observed'), 'bootstrap_error': BOOTSTRAP_ERROR})


In [ ]:
# 3. Install the locked minimal runtime
import sys
if BOOTSTRAP_ERROR is None:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers==0.35.2', 'transformers==4.49.0', 'accelerate==1.4.0', 'imageio==2.37.0', 'imageio-ffmpeg==0.6.0', 'ftfy==6.3.1', 'safetensors==0.5.3'], check=True)
        import torch, diffusers, transformers, accelerate
        if not torch.cuda.is_available():
            raise RuntimeError('A real Colab CUDA runtime is required')
    except Exception as exc:
        BOOTSTRAP_ERROR = f'runtime setup failure: {type(exc).__name__}: {exc}'
print({'python': sys.version, 'gpu': torch.cuda.get_device_name(0) if BOOTSTRAP_ERROR is None else None, 'bootstrap_error': BOOTSTRAP_ERROR})


In [ ]:
# 4. Call only the repository v2 CLI
import hashlib, json, platform
CONFIG = REPO / 'configs/gpu_carrier_propagation_v2.json'
if BOOTSTRAP_ERROR is None:
    config_sha = hashlib.sha256(CONFIG.read_bytes()).hexdigest()
    cmd = [sys.executable, 'experiments/run_gpu_carrier_propagation_v2.py', '--config', str(CONFIG), '--output-dir', str(LOCAL_RUN), '--expected-commit', EXPECTED_COMMIT]
    print({'config': str(CONFIG), 'config_sha256': config_sha, 'command': cmd})
    process = subprocess.run(cmd, cwd=REPO, text=True, capture_output=True)
    print(process.stdout[-4000:])
    if process.stderr:
        print(process.stderr[-4000:])
    RETURN_CODE = process.returncode
else:
    RETURN_CODE = 2
    LOCAL_RUN.mkdir(parents=True, exist_ok=True)
    (LOCAL_RUN / 'artifacts').mkdir(exist_ok=True)
    failure = {'evidence_kind': 'gpu_carrier_propagation_v2_bootstrap_failure', 'gate_pass': False, 'reason': BOOTSTRAP_ERROR, 'method_claim': False}
    for name, payload in [('environment.json', {'python': platform.python_version()}), ('git_state.json', {'expected_commit': EXPECTED_COMMIT, 'verified': False}), ('config.json', {'available': CONFIG.exists()}), ('metrics.json', failure), ('gate_decision.json', {'implementer_decision': 'GATE_FAIL', 'auditor_decision': 'PENDING', 'reason': BOOTSTRAP_ERROR})]:
        (LOCAL_RUN / name).write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    (LOCAL_RUN / 'artifacts' / 'failure.json').write_text(json.dumps(failure, indent=2) + '\n', encoding='utf-8')
    (LOCAL_RUN / 'command.txt').write_text('bootstrap failed before repository CLI\n', encoding='utf-8')
    (LOCAL_RUN / 'stdout.log').write_text('', encoding='utf-8')
    (LOCAL_RUN / 'stderr.log').write_text(BOOTSTRAP_ERROR + '\n', encoding='utf-8')
    (LOCAL_RUN / 'README.md').write_text('# GPU carrier propagation v2 bootstrap failure\n\nNo bridge or method claim.\n', encoding='utf-8')


In [ ]:
# 5. Complete, display, and copy one immutable package
import hashlib, json, shutil
checksum_file = LOCAL_RUN / 'checksums.sha256'
if not checksum_file.exists():
    targets = sorted(path for path in LOCAL_RUN.rglob('*') if path.is_file() and path != checksum_file)
    checksum_file.write_text(''.join(f"{hashlib.sha256(path.read_bytes()).hexdigest()}  {path.relative_to(LOCAL_RUN).as_posix()}\n" for path in targets), encoding='utf-8')
local_archive = LOCAL_RUN.with_suffix('.tar.gz')
if not local_archive.exists():
    local_archive = Path(shutil.make_archive(str(LOCAL_RUN), 'gztar', root_dir=LOCAL_RUN.parent, base_dir=LOCAL_RUN.name))
DRIVE_PARENT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN = DRIVE_PARENT / RUN_NAME
if DRIVE_RUN.exists():
    raise RuntimeError(f'refusing to overwrite existing Drive run: {DRIVE_RUN}')
shutil.copytree(LOCAL_RUN, DRIVE_RUN)
drive_archive = DRIVE_PARENT / local_archive.name
if drive_archive.exists():
    raise RuntimeError(f'refusing to overwrite existing Drive archive: {drive_archive}')
shutil.copy2(local_archive, drive_archive)
decision = json.loads((LOCAL_RUN / 'metrics.json').read_text(encoding='utf-8'))
print({'drive_run': str(DRIVE_RUN), 'drive_archive': str(drive_archive), 'return_code': RETURN_CODE, 'decision': decision})
if RETURN_CODE != 0:
    raise RuntimeError('GPU carrier propagation v2 failed after preserving its Drive package')
